# Análisis de Red de Tiendas RetailNow

## 1 y 2. Preparar el entorno e importar librerías

In [ ]:
import pandas as pd
import numpy as np

## 3. Cargar y procesar los datos

In [ ]:
# Definimos las rutas absolutas según el requerimiento
ruta_sales = '/workspace/sales.csv'
ruta_inventories = '/workspace/inventories.csv'
ruta_satisfaction = '/workspace/satisfaction.csv'

import os
# Si estamos probando en un entorno local, ajustamos la ruta:
if not os.path.exists('/workspace'):
    ruta_sales = '/home/jlino/Cloud/UNIR/Cursos/Programacion_Python/repo/sales.csv'
    ruta_inventories = '/home/jlino/Cloud/UNIR/Cursos/Programacion_Python/repo/inventories.csv'
    ruta_satisfaction = '/home/jlino/Cloud/UNIR/Cursos/Programacion_Python/repo/satisfaction.csv'

# Cargar los CSV
sales_df = pd.read_csv(ruta_sales)
inventories_df = pd.read_csv(ruta_inventories)
satisfaction_df = pd.read_csv(ruta_satisfaction)

# Limpiar los datos eliminando valores nulos
sales_df = sales_df.dropna()
inventories_df = inventories_df.dropna()
satisfaction_df = satisfaction_df.dropna()

print("Datos de ventas cargados:")
display(sales_df.head())

## 4. Exploración de datos (Pandas)

In [ ]:
# Calcular los ingresos totales por tienda (Cantidad_Vendida * Precio_Unitario)
sales_df['Total_Ventas'] = sales_df['Cantidad_Vendida'] * sales_df['Precio_Unitario']

# Calcular las ventas totales por producto y por tienda
ventas_por_producto_tienda = sales_df.groupby(['ID_Tienda', 'Producto']).agg(
    Total_Cantidad_Vendida=('Cantidad_Vendida', 'sum'),
    Ingresos_Totales=('Total_Ventas', 'sum')
).reset_index()

print("Ventas totales por producto y por tienda:")
display(ventas_por_producto_tienda)

# Ventas totales (ingresos) por tienda
ventas_totales_tienda = sales_df.groupby('ID_Tienda')['Total_Ventas'].sum().reset_index()
print("\nVentas totales por tienda:")
display(ventas_totales_tienda)

# Resumen estadístico de las ventas
print("\nResumen estadístico de las ventas:")
display(sales_df.describe())

## 5. Análisis de inventarios (Pandas)

In [ ]:
# Agrupar las ventas por tienda y producto para sumarlas
ventas_agrupadas = sales_df.groupby(['ID_Tienda', 'Producto'])['Cantidad_Vendida'].sum().reset_index()

# Hacer merge de los inventarios con las cantidades vendidas
inventarios_vs_ventas = pd.merge(inventories_df, ventas_agrupadas, on=['ID_Tienda', 'Producto'], how='left')

# Si algún producto en el inventario no tuvo ventas, rellenar el NaN con 0
inventarios_vs_ventas['Cantidad_Vendida'] = inventarios_vs_ventas['Cantidad_Vendida'].fillna(0)

# Calcular la rotación de inventario (Cantidad Vendida / Stock Disponible) * 100
inventarios_vs_ventas['Rotacion_Inventario_%'] = (inventarios_vs_ventas['Cantidad_Vendida'] / inventarios_vs_ventas['Stock_Disponible']) * 100

print("Análisis de rotación de inventarios por producto y tienda:")
display(inventarios_vs_ventas)

# Filtrar las tiendas con niveles críticos (rotación menor al 10%)
inventario_critico = inventarios_vs_ventas[inventarios_vs_ventas['Rotacion_Inventario_%'] < 10]
print("\nTiendas con niveles críticos de inventario (< 10% de ventas respecto al stock):")
display(inventario_critico)

## 6. Satisfacción del cliente (Pandas)

In [ ]:
# Relacionar la satisfacción del cliente con las ventas totales por tienda
rendimiento_y_satisfaccion = pd.merge(ventas_totales_tienda, satisfaction_df, on='ID_Tienda')

print("Rendimiento de ventas y satisfacción por tienda:")
display(rendimiento_y_satisfaccion)

# Filtrar tiendas con niveles bajos de satisfacción (< 60%)
tiendas_baja_satisfaccion = rendimiento_y_satisfaccion[rendimiento_y_satisfaccion['Satisfacción_Promedio'] < 60]

print("\nTiendas con niveles bajos de satisfacción (< 60%):")
display(tiendas_baja_satisfaccion)

# Recomendación: Las tiendas con baja satisfacción requieren investigación sobre la calidad del servicio al cliente.

## 7. Operaciones con Numpy

In [ ]:
# Extraer la columna de ventas a un array de Numpy
ventas_array = sales_df['Total_Ventas'].to_numpy()

# Calcular mediana de las ventas totales
mediana_ventas = np.median(ventas_array)
print(f"Mediana de las ventas totales: {mediana_ventas}")

# Calcular desviación estándar de las ventas
desviacion_estandar_ventas = np.std(ventas_array)
print(f"Desviación estándar de las ventas totales: {desviacion_estandar_ventas:.2f}")

# Simulación de proyecciones futuras (generando arrays aleatorios)
np.random.seed(42)
media_ventas = np.mean(ventas_array)

# Generar 10 proyecciones de ventas futuras usando una distribución normal
proyecciones_futuras = np.random.normal(loc=media_ventas, scale=desviacion_estandar_ventas, size=10)

print("\nProyecciones aleatorias de ventas futuras:")
print(proyecciones_futuras)